# CARE v6 · Wind Farm A · прогноз H7

Используйте CPU runtime. По умолчанию `RUN_TRAINING = False`. Первый запуск проверяет файлы, дубли, независимость разбиения и покрытие H7. Notebook открывает фиксированное разбиение по целым турбинам. Test закреплён за турбиной 21, пороги модели выбираются на validation.

Скопируйте папку проекта в `MyDrive/scada-pdm-mvp`. Если Wind Farm A уже скачан, загрузите подпапку `data/CARE_To_Compare/Wind Farm A/` вместе с `datasets/`, `event_info.csv` и `feature_description.csv`. При отсутствии файлов ячейка 3 запрашивает только файлы Wind Farm A через HTTP Range из [CARE v6 на Zenodo](https://zenodo.org/records/15846963).

Выполните ячейки 1–5 по порядку. Если H7 получил `eligible`, для исследовательского запуска укажите путь к проверенному кешу и включите `RUN_TRAINING`, затем запустите ячейки 1–7. Прогноз — некалиброванный score для отобранных событий CARE. В тесте одна турбина, поэтому метрики характеризуют только этот эксперимент.


In [ ]:
# 1. Настройки и подключение Google Drive
from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

USE_DRIVE = True
PROJECT_NAME = "scada-pdm-mvp"
CONFIG_NAME = "care_h7.json"
EXISTING_CACHE = ""  # Абсолютный путь к готовому кешу. Пустая строка создаёт новый.
RUN_TRAINING = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive")
else:
    root = Path("/content")

PROJECT = root / PROJECT_NAME
CONFIG = PROJECT / "configs" / CONFIG_NAME
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SESSION = root / "scada-pdm-runs" / "care-h7" / RUN_ID
CACHE = Path(EXISTING_CACHE) if EXISTING_CACHE else SESSION / "cache"
SOURCE_REPORT = SESSION / "sources.json"
OUTPUT = SESSION / ("training" if RUN_TRAINING else "audit")
PYTHON = Path("/content/scada-pdm-venv/bin/python")

if not CONFIG.is_file():
    raise FileNotFoundError(f"Укажите каталог проекта и конфигурацию: {CONFIG}")
if sys.version_info[:2] not in {(3, 11), (3, 12), (3, 13), (3, 14)}:
    raise RuntimeError("Для зафиксированных зависимостей требуется Python 3.11–3.14")


In [ ]:
# 2. Установка CPU-зависимостей
if not PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "venv", str(PYTHON.parent.parent)], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(PROJECT / "requirements-ml.txt")], check=True)

def cli(*args, check=True):
    command = [str(PYTHON), "-m", "scada_pdm.cli", *map(str, args)]
    return subprocess.run(command, cwd=PROJECT, check=check)


In [ ]:
# 3. Проверка локального CARE или адресная загрузка Wind Farm A
if EXISTING_CACHE:
    if not (CACHE / "coverage.json").is_file():
        raise FileNotFoundError(f"Отсутствует отчёт готового кеша: {CACHE / 'coverage.json'}")
    print("Будет использован готовый кеш", CACHE)
else:
    cli("download", "--config", CONFIG, "--output", SOURCE_REPORT)
    print("Источник", SOURCE_REPORT)


In [ ]:
# 4. Проверка телеметрии, сопоставление событий, H7 и признаки
if not EXISTING_CACHE:
    cli("prepare", "--config", CONFIG, "--cache", CACHE)
coverage = json.loads((CACHE / "coverage.json").read_text(encoding="utf-8"))

import pandas as pd
from IPython.display import display

print("Проверено файлов", coverage["signals"]["events"], "пар", coverage["pairs_examined"])
print("Найдено пар с пересечением", coverage["overlap_pairs"])
print("Групп разбиения", len(coverage["dependency_groups"]))
rows = {}
for stage in ("raw_h7", "h7_after_reconciliation", "h7_with_feature_history"):
    entry = coverage[stage]
    rows[stage] = {
        "positive": entry["positive"],
        "negative": entry["negative"],
        "episodes": len(entry["positive_episode_ids"]),
    }
display(pd.DataFrame.from_dict(rows, orient="index"))
display(pd.DataFrame(coverage["fault_groups"]))
print("Покрытие и связи файлов", CACHE / "coverage.json", CACHE / "overlaps.json")
print("Каталог кеша", CACHE)


In [ ]:
# 5. Независимость групп и пригодность H7. Обучение включается явно.
flags = [] if RUN_TRAINING else ["--audit-only"]
cli("train", "--config", CONFIG, "--cache", CACHE, "--output", OUTPUT, *flags)
report = json.loads((OUTPUT / "metrics.json").read_text(encoding="utf-8"))
h7 = report["horizons"]["h7"]
print("H7", h7["status"], h7.get("reasons", []))
print("Пересечения между выборками", report.get("raw_fold_audit", {}))
display(pd.DataFrame.from_dict(h7["coverage"], orient="index"))
print("Каталог отчёта", OUTPUT)


In [ ]:
# 6. Метрики и кривые PR/ROC
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Image

metrics_file = OUTPUT / "metrics.csv"
if metrics_file.is_file():
    display(pd.read_csv(metrics_file))
for horizon, state in report["horizons"].items():
    if state["status"] != "trained":
        display(pd.DataFrame.from_dict(state["coverage"], orient="index"))
        continue
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    for model in state["models"]:
        curves = pd.read_csv(OUTPUT / horizon / model / "test_curves.csv")
        for ax, kind, labels in zip(axes, ("PR", "ROC"), (("Recall", "Precision"), ("FPR", "TPR"))):
            values = curves[curves["curve"] == kind]
            ax.plot(values["x"], values["y"], label=model)
            ax.set(xlabel=labels[0], ylabel=labels[1], title=f"{horizon.upper()} · test {kind}")
            ax.legend()
    figure.tight_layout()
    figure.savefig(OUTPUT / f"{horizon}_test_curves.png", dpi=140)
    plt.show()


In [ ]:
# 7. Gain, global/local TreeSHAP и история датчиков
for horizon, state in report["horizons"].items():
    if state["status"] != "trained":
        continue
    outputs = OUTPUT / horizon / "lightgbm"
    print(horizon.upper())
    display(pd.read_csv(outputs / "sensor_importance.csv").head(10))
    for plot in ("gain.png", "shap_global.png", "shap_local.png"):
        display(Image(filename=str(outputs / plot)))
    print("История риск-скора и датчиков:", outputs / "risk_sensor_history.parquet")
print("Артефакты запуска:", OUTPUT)

if not any(state["status"] == "trained" for state in report["horizons"].values()):
    print("Для этого запуска сохранён отчёт покрытия. Файлы SHAP появятся у обученных горизонтов.")
